<a href="https://colab.research.google.com/github/csu-techhub/QOS-Labs/blob/main/Assignment_20_Node.ipynb" target="_parent">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>


# Assignment 1 — QAOA with 20-Node Max-Cut
**Quantum Optimization and Simulation — QAOA Laboratory Series**

> Qiskit displays measured bitstrings in the order `q_(n-1)...q_0`. When we discuss graph nodes, this notebook often converts them to `q_0...q_(n-1)` using `q0_first(...)`.

**Problem instance:** a fixed 20-node, 3-regular graph with 30 edges. Its **known optimum Max-Cut value is 26**, which we use as the yardstick for every result in this lab.

> 20 qubits is much heavier than the 5-qubit graph used earlier. The ideal-simulator parts take a minute or two in Colab; the noisy simulation in Part D takes about **6–8 minutes**.

## Learning goals
- Close the hybrid quantum-classical loop on a 20-qubit Max-Cut instance.
- Use COBYLA to optimize $(\gamma, \beta)$ for $p=1$ and $p=2$, and compare their convergence.
- Judge results with the **approximation ratio** $r = \langle C \rangle / C_{\max}$ against a known optimum.
- See how hardware noise (FakeSherbrooke model) erodes QAOA performance, and why deeper circuits suffer more.

In [ ]:
# Run this once at the beginning of a fresh Google Colab session.
%pip -q install "pylatexenc" "qiskit~=2.5" "qiskit-aer~=0.17" "qiskit-optimization~=0.7" "qiskit-ibm-runtime~=0.46"

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from collections import Counter

from qiskit import QuantumCircuit
from qiskit.visualization import plot_histogram
from qiskit_aer.primitives import SamplerV2

SEED = 123
SHOTS = 2048
sampler = SamplerV2(default_shots=SHOTS, seed=SEED)

def run_counts(qc, shots=SHOTS):
    "Run a measured circuit with Aer SamplerV2 and return counts."
    result = sampler.run([qc], shots=shots).result()
    return result[0].data.meas.get_counts()

def q0_first(qiskit_bitstring):
    "Convert Qiskit's displayed q_(n-1)...q_0 bitstring to q_0...q_(n-1)."
    return qiskit_bitstring.replace(" ", "")[::-1]

In [ ]:
from scipy.optimize import minimize

# 20-node 3-regular graph (30 edges), generated once with
# nx.random_regular_graph(3, 20, seed=0) and written out explicitly so it
# never changes across NetworkX versions.
N_NODES = 20
edges20 = [
    (0, 1), (0, 9), (0, 15), (1, 4), (1, 18), (2, 4), (2, 6), (2, 17),
    (3, 6), (3, 7), (3, 11), (4, 14), (5, 7), (5, 10), (5, 11), (6, 11),
    (7, 10), (8, 16), (8, 18), (8, 19), (9, 14), (9, 16), (10, 19), (12, 13),
    (12, 16), (12, 17), (13, 17), (13, 19), (14, 15), (15, 18),
]
MAXCUT_OPT = 26        # known optimum (exhaustive search over all 2^20 partitions)
RANDOM_CUT = len(edges20) / 2   # expected cut of a uniformly random partition

def cut_value_q0first(bitstring, edges):
    return sum(bitstring[i] != bitstring[j] for i, j in edges)

def expected_cut_from_counts(counts, edges):
    total = sum(counts.values())
    return sum(
        cut_value_q0first(q0_first(k), edges) * v / total
        for k, v in counts.items()
    )

def build_qaoa(gammas, betas):
    p = len(gammas)
    qc = QuantumCircuit(N_NODES)
    qc.h(range(N_NODES))

    for layer in range(p):
        gamma = gammas[layer]
        beta = betas[layer]

        for i, j in edges20:
            qc.cx(i, j)
            qc.rz(-gamma, j)
            qc.cx(i, j)

        for q in range(N_NODES):
            qc.rx(2 * beta, q)

    qc.measure_all()
    return qc

## Part A — Meet the 20-node graph
Every node has exactly three neighbors (a *3-regular* graph). Look at the picture before running any quantum code: Max-Cut asks us to color each node one of two colors so that as many edges as possible join nodes of **different** colors.

In [ ]:
G = nx.Graph()
G.add_nodes_from(range(N_NODES))
G.add_edges_from(edges20)
pos = nx.kamada_kawai_layout(G)

print(f"nodes = {G.number_of_nodes()}, edges = {G.number_of_edges()}")
print(f"node degrees = {sorted(set(dict(G.degree()).values()))}")
print(f"bipartite?   = {nx.is_bipartite(G)}")
print(f"known optimum cut = {MAXCUT_OPT} of {len(edges20)} edges")

plt.figure(figsize=(6.5, 6.5))
nx.draw(G, pos, with_labels=True, node_color="#9ecae1", edgecolors="#2b6c9e",
        node_size=500, font_size=9, edge_color="#555555")
plt.title("20-node 3-regular Max-Cut instance")
plt.show()

## Part B — Define the optimization objectives
Each objective builds a QAOA circuit, samples it, and returns the **negative** average cut. Every call also records the average cut in a history list so we can plot convergence.

In [ ]:
history_p1 = []
history_p2 = []

def objective_p1(params):
    gamma, beta = params
    qc = build_qaoa([gamma], [beta])
    counts = run_counts(qc, shots=1024)
    avg_cut = expected_cut_from_counts(counts, edges20)
    history_p1.append(avg_cut)
    return -avg_cut

def objective_p2(params):
    gammas = params[:2]
    betas = params[2:]
    qc = build_qaoa(gammas, betas)
    counts = run_counts(qc, shots=1024)
    avg_cut = expected_cut_from_counts(counts, edges20)
    history_p2.append(avg_cut)
    return -avg_cut

## Part C — Run COBYLA for p=1 and p=2 and compare convergence

In [ ]:
history_p1.clear()
result_p1 = minimize(
    objective_p1,
    x0=np.array([0.3, 0.6]),
    method="COBYLA",
    options={"maxiter": 40, "rhobeg": 0.5}
)
print(f"p=1: gamma, beta = {np.round(result_p1.x, 3)}")
print(f"     average cut ≈ {-result_p1.fun:.2f}   approximation ratio ≈ {-result_p1.fun / MAXCUT_OPT:.3f}   ({len(history_p1)} evaluations)")

In [ ]:
history_p2.clear()
result_p2 = minimize(
    objective_p2,
    x0=np.array([0.5, 0.8, 0.3, 0.5]),   # [gamma1, gamma2, beta1, beta2]
    method="COBYLA",
    options={"maxiter": 60}
)
print(f"p=2: gammas = {np.round(result_p2.x[:2], 3)}, betas = {np.round(result_p2.x[2:], 3)}")
print(f"     average cut ≈ {-result_p2.fun:.2f}   approximation ratio ≈ {-result_p2.fun / MAXCUT_OPT:.3f}   ({len(history_p2)} evaluations)")

In [ ]:
plt.figure(figsize=(8, 4.5))
plt.plot(range(1, len(history_p1) + 1), history_p1, marker="o", ms=3, label="p = 1")
plt.plot(range(1, len(history_p2) + 1), history_p2, marker="s", ms=3, label="p = 2")
plt.axhline(MAXCUT_OPT, color="red", linestyle="--", label=f"known optimum = {MAXCUT_OPT}")
plt.axhline(RANDOM_CUT, color="gray", linestyle=":", label=f"random guess = {RANDOM_CUT:.0f}")
plt.xlabel("Objective evaluation (one circuit execution each)")
plt.ylabel("Average cut")
plt.title("COBYLA convergence on the 20-node graph: p=1 vs p=2")
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.show()

**Expected:** both curves climb from the starting point, with shot noise making them non-monotonic. $p=1$ should settle around an approximation ratio of 0.75–0.78 (theory guarantees at least about 0.692 for 3-regular graphs). $p=2$ climbs higher, roughly 0.82–0.84, but needs more evaluations because it has twice as many parameters.

Now look at the best partition the optimized $p=2$ circuit actually produced:

In [ ]:
qc_p1 = build_qaoa(result_p1.x[:1], result_p1.x[1:])
qc_p2 = build_qaoa(result_p2.x[:2], result_p2.x[2:])

final_counts = run_counts(qc_p2, shots=SHOTS)
best = max((q0_first(k) for k in final_counts), key=lambda s: cut_value_q0first(s, edges20))
best_cut = cut_value_q0first(best, edges20)

node_colors = ["#f4a261" if b == "1" else "#2a9d8f" for b in best]
cut_edges = [(i, j) for i, j in edges20 if best[i] != best[j]]
uncut_edges = [(i, j) for i, j in edges20 if best[i] == best[j]]

plt.figure(figsize=(6.5, 6.5))
nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=500, edgecolors="black")
nx.draw_networkx_labels(G, pos, font_size=9)
nx.draw_networkx_edges(G, pos, edgelist=cut_edges, edge_color="black", width=2)
nx.draw_networkx_edges(G, pos, edgelist=uncut_edges, edge_color="red", style="dashed", width=1.5)
plt.title(f"Best sampled partition (p=2): cut = {best_cut} / optimum {MAXCUT_OPT}\nsolid = cut edge, red dashed = uncut edge")
plt.axis("off")
plt.show()
print("bitstring (q0 first):", best)

### Optional
1. Change the $p=1$ initial point to `[1.5, 1.5]` in the first cell in Part C and rerun. Do you get the same final parameters? A similar final cut value?
2. **Warm start.** A common trick is to start $p=2$ from the optimized $p=1$ angles, e.g. `x0 = [γ*, γ*, β*, β*]`, as in the cell below. Run the cell and compare the final cut and the number of evaluations with the cold start above.
3. Look at the convergence chart. On real hardware every point on these curves is one circuit execution (one queue wait on the Open plan). Was the gain from $p=2$ worth its extra evaluations?
4. In the partition plot, count the red dashed edges. How does that number relate to the cut value?

In [ ]:
# Optional: warm-start p=2 from the p=1 solution.
g1, b1 = result_p1.x
history_p2.clear()
result_warm = minimize(objective_p2, x0=np.array([g1, g1, b1, b1]), method="COBYLA", options={"maxiter": 60})
print(-result_warm.fun / MAXCUT_OPT, len(history_p2))

## Part D — The same circuits under a realistic noise model (FakeSherbrooke)
`FakeSherbrooke` is a snapshot of a 127-qubit IBM Eagle processor: its heavy-hex connectivity, native gates (ECR, √X, RZ), and calibrated error rates. Two things happen to our circuit:
1. **Transpilation.** The graph's edges do not match the chip's wiring, so the transpiler inserts SWAPs and rewrites every gate into the native set. The circuit gets much deeper.
2. **Noise.** Every gate and measurement now has a chance of error, and the errors pile up with depth.

**Why we do not run COBYLA here:** simulating noise on 20 qubits is done shot by shot: roughly 0.6 s per shot for $p=1$ and over 2 s per shot for $p=2$ in Colab. One 1024-shot evaluation would take 10–40 minutes, and a full optimization would take many hours. So we do what we would do on hardware: optimize on the ideal simulator (Part C), then **evaluate the final parameters once** under noise.

In [ ]:
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke

fake_backend  = FakeSherbrooke()
noisy_backend = AerSimulator.from_backend(fake_backend)

noisy_pm      = generate_preset_pass_manager(backend=noisy_backend,
                                             optimization_level=3,
                                             seed_transpiler=SEED)
noisy_sampler = SamplerV2.from_backend(noisy_backend, seed=SEED)

NOISY_SHOTS = 128   # noisy 20-qubit simulation is slow: ~1–2 min for p=1, ~5 min for p=2

def run_counts_noisy(isa_qc, shots=NOISY_SHOTS):
    "Run an already-transpiled (ISA) circuit with the FakeSherbrooke noise model."
    result = noisy_sampler.run([isa_qc], shots=shots).result()
    return result[0].data.meas.get_counts()

In [ ]:
isa_p1 = noisy_pm.run(qc_p1)
isa_p2 = noisy_pm.run(qc_p2)

for name, qc, isa in [("p=1", qc_p1, isa_p1), ("p=2", qc_p2, isa_p2)]:
    print(f"{name}: logical depth {qc.depth():>4}, CNOTs {qc.count_ops().get('cx', 0):>3}  ->  "
          f"FakeSherbrooke depth {isa.depth():>4}, ECRs {isa.count_ops().get('ecr', 0):>3}")

In [ ]:
import time

ideal_counts = {"p=1": run_counts(qc_p1), "p=2": run_counts(qc_p2)}
noisy_counts = {}
for name, isa in [("p=1", isa_p1), ("p=2", isa_p2)]:
    t0 = time.time()
    noisy_counts[name] = run_counts_noisy(isa)
    print(f"{name} noisy simulation finished in {time.time() - t0:.0f} s")

summary = {}
print(f"\n{'':5} {'ideal <cut>':>12} {'noisy <cut>':>12} {'ideal r':>9} {'noisy r':>9}")
for name in ["p=1", "p=2"]:
    ci = expected_cut_from_counts(ideal_counts[name], edges20)
    cn = expected_cut_from_counts(noisy_counts[name], edges20)
    summary[name] = (ci, cn)
    print(f"{name:5} {ci:12.2f} {cn:12.2f} {ci/MAXCUT_OPT:9.3f} {cn/MAXCUT_OPT:9.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# Left: average cut, ideal vs noisy
x = np.arange(2)
w = 0.35
axes[0].bar(x - w/2, [summary[k][0] for k in ["p=1", "p=2"]], w, label="ideal Aer")
axes[0].bar(x + w/2, [summary[k][1] for k in ["p=1", "p=2"]], w, label="FakeSherbrooke")
axes[0].axhline(MAXCUT_OPT, color="red", linestyle="--", label="known optimum")
axes[0].axhline(RANDOM_CUT, color="gray", linestyle=":", label="random guess")
axes[0].set_xticks(x, ["p = 1", "p = 2"])
axes[0].set_ylabel("Average cut"); axes[0].set_ylim(0, 33)
axes[0].set_title("Optimized QAOA: ideal vs noisy"); axes[0].legend(loc="upper center", ncol=2, fontsize=9)

# Right: distribution of sampled cut values for p=2
def cut_distribution(counts):
    total = sum(counts.values())
    dist = Counter()
    for k, v in counts.items():
        dist[cut_value_q0first(q0_first(k), edges20)] += v / total
    return dist

di, dn = cut_distribution(ideal_counts["p=2"]), cut_distribution(noisy_counts["p=2"])
vals = np.arange(8, MAXCUT_OPT + 1)
axes[1].bar(vals - 0.2, [di[v] for v in vals], 0.4, label="ideal Aer")
axes[1].bar(vals + 0.2, [dn[v] for v in vals], 0.4, label="FakeSherbrooke")
axes[1].axvline(MAXCUT_OPT, color="red", linestyle="--")
axes[1].set_xlabel("Cut value"); axes[1].set_ylabel("Fraction of shots")
axes[1].set_title("Sampled cut values, p = 2"); axes[1].legend()
plt.tight_layout(); plt.show()

## Questions
1. Why do we return `-avg_cut` to SciPy?
2. Why can two different parameter sets give similar solution quality?